In [7]:
import os
import pathlib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, r2_score

In [11]:
def get_preds(models, base_path, split, test_set):
    all_preds = []
    for i, model in enumerate(models):
        model_path = os.path.join(base_path, split, model)
        model_dir = pathlib.Path(model_path).parent
        model_prediction = pd.read_csv(f'{model_dir}/{test_set}')
        all_preds.append(model_prediction['unscaled_prediction'].rename(str(i)))
    predictions = pd.concat(all_preds, axis=1)
    model_prediction['mean_predictions'] = predictions.mean(axis=1)
    return predictions, model_prediction

def get_performance(y_true, predictions):
    # 1. Initialize a dictionary to store individual metrics
    metrics_dict = {'RMSE': [], 'R2': [], 'Spearman_Rho': []}
    
    # 2. Calculate metrics for each model
    for i in range(len(models)):
        y_pred = predictions[str(i)]
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        rho, _ = spearmanr(y_true, y_pred)
        
        metrics_dict['RMSE'].append(rmse)
        metrics_dict['R2'].append(r2)
        metrics_dict['Spearman_Rho'].append(rho)
    
    # 3. Convert to a DataFrame where rows = models, columns = metrics
    df_metrics = pd.DataFrame(metrics_dict, index=[f'Model_{i}' for i in range(len(models))])
    
    # 4. Compute Summary Statistics
    summary_stats = pd.DataFrame({
        'Median': df_metrics.median(),
        'Mean': df_metrics.mean(),
        'Std_Dev': df_metrics.std()
    })
    return summary_stats

In [30]:
def get_mean_preds(models, base_path, split):
    all_preds = []
    for i, model in enumerate(models):
        model_path = os.path.join(base_path, split, model)
        model_dir = pathlib.Path(model_path).parent
        model_prediction = pd.read_csv(f'{model_dir}/saifudeen_train_preds.csv')
        all_preds.append(model_prediction['unscaled_prediction'].rename(str(i)))
    predictions = pd.concat(all_preds, axis=1)
    model_prediction['mean_predictions'] = predictions.mean(axis=1)
    return model_prediction

In [12]:
current_dir = pathlib.Path.cwd()
base_path = current_dir / 'models/'
saifudeen_test = 'saifudeen_test_preds.csv'
cluster_test = 'cluster_test_preds.csv'

In [20]:
split = 'cluster_split_base_set/'
models = ['1/best-51-val_rmse.pt',
          '2/best-58-val_rmse.pt',
          '3/best-93-val_rmse.pt',
          '4/best-91-val_rmse.pt',
          '5/best-96-val_rmse.pt',
          '6/best-92-val_rmse.pt',
          '7/best-69-val_rmse.pt',
          '8/best-60-val_rmse.pt',
          '9/best-75-val_rmse.pt',
          '10/best-51-val_rmse.pt'
         ]

predictions, model_prediction = get_preds(models, base_path, split, saifudeen_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'saifudeen (external test) performance: \n {summary_stats}\n')

predictions, model_prediction = get_preds(models, base_path, split, cluster_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'cluster (internal test) performance: \n {summary_stats}\n')

saifudeen (external test) performance: 
                 Median      Mean   Std_Dev
RMSE          1.032243  1.038743  0.024722
R2            0.112756  0.101089  0.043592
Spearman_Rho  0.536438  0.538044  0.011714

cluster (internal test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.833874  0.832961  0.006616
R2            0.498825  0.499894  0.007928
Spearman_Rho  0.705354  0.704889  0.005184



In [47]:
split = 'cluster_split_base_rank_set/'
models = ['1/best-68-val_rmse.pt',
          '2/best-93-val_rmse.pt',
          '3/best-74-val_rmse.pt',
          '4/best-84-val_rmse.pt',
          '5/best-83-val_rmse.pt',
          '6/best-71-val_rmse.pt',
          '7/best-79-val_rmse.pt',
          '8/best-52-val_rmse.pt',
          '9/best-75-val_rmse.pt',
          '10/best-74-val_rmse.pt'
         ]

predictions, model_prediction = get_preds(models, base_path, split, saifudeen_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'saifudeen (external test) performance: \n {summary_stats}\n')

predictions, model_prediction = get_preds(models, base_path, split, cluster_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'cluster (internal test) performance: \n {summary_stats}\n')

base_rank_saif_train_mean = get_mean_preds(models, base_path, split)
base_rank_saif_train_mean = base_rank_saif_train_mean[base_rank_saif_train_mean['mut'] == 'WT']

saifudeen (external test) performance: 
                 Median      Mean   Std_Dev
RMSE          1.027232  1.031623  0.016070
R2            0.121349  0.113628  0.027736
Spearman_Rho  0.538562  0.538945  0.008182

cluster (internal test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.834064  0.833028  0.005095
R2            0.498597  0.499825  0.006111
Spearman_Rho  0.706050  0.705660  0.003296



In [22]:
split = 'cluster_split_ext_rank_set/'
models = ['1/best-39-val_rmse.pt',
          '2/best-77-val_rmse.pt',
          '3/best-52-val_rmse.pt',
          '4/best-66-val_rmse.pt',
          '5/best-72-val_rmse.pt',
          '6/best-60-val_rmse.pt',
          '7/best-88-val_rmse.pt',
          '8/best-98-val_rmse.pt',
          '9/best-64-val_rmse.pt',
          '10/best-71-val_rmse.pt',
         ]


predictions, model_prediction = get_preds(models, base_path, split, saifudeen_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'saifudeen (external test) performance: \n {summary_stats}\n')

predictions, model_prediction = get_preds(models, base_path, split, cluster_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'cluster (internal test) performance: \n {summary_stats}\n')

saifudeen (external test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.883511  0.874458  0.018728
R2            0.350013  0.363005  0.027053
Spearman_Rho  0.665406  0.663910  0.013738

cluster (internal test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.828740  0.827020  0.006010
R2            0.504976  0.507006  0.007147
Spearman_Rho  0.709775  0.710216  0.004118



In [53]:
split = 'cluster_split_ext_rank_cont_set/'
models = ['1/best-60-val_rmse.pt',
          '2/best-61-val_rmse.pt',
          '3/best-67-val_rmse.pt',
          '4/best-92-val_rmse.pt',
          '5/best-45-val_rmse.pt',
          '6/best-87-val_rmse.pt',
          '7/best-43-val_rmse.pt',
          '8/best-72-val_rmse.pt',
          '9/best-80-val_rmse.pt',
          '10/best-50-val_rmse.pt',
         ]


predictions, model_prediction = get_preds(models, base_path, split, saifudeen_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'saifudeen (external test) performance: \n {summary_stats}\n')

predictions, model_prediction = get_preds(models, base_path, split, cluster_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'cluster (internal test) performance: \n {summary_stats}\n')


ext_rank_cont_saif_train_mean = get_mean_preds(models, base_path, split)
ext_rank_cont_saif_train_mean = ext_rank_cont_saif_train_mean[ext_rank_cont_saif_train_mean['mut'] == 'WT']

saifudeen (external test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.864549  0.863079  0.012253
R2            0.377615  0.379619  0.017579
Spearman_Rho  0.683017  0.684000  0.010242

cluster (internal test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.825290  0.827015  0.006693
R2            0.509090  0.507007  0.007993
Spearman_Rho  0.711787  0.710530  0.005906



In [52]:
split = 'cluster_split_ext_rank_cont_bin_set/'
models = ['1/best-39-val_rmse.pt',
          '2/best-90-val_rmse.pt',
          '3/best-83-val_rmse.pt',
          '4/best-56-val_rmse.pt',
          '5/best-67-val_rmse.pt',
          '6/best-66-val_rmse.pt',
          '7/best-75-val_rmse.pt',
          '8/best-70-val_rmse.pt',
          '9/best-65-val_rmse.pt',
         ]


predictions, model_prediction = get_preds(models, base_path, split, saifudeen_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'saifudeen (external test) performance: \n {summary_stats}\n')

predictions, model_prediction = get_preds(models, base_path, split, cluster_test)
summary_stats = get_performance(model_prediction['value'], predictions)
print(f'cluster (internal test) performance: \n {summary_stats}\n')


saifudeen (external test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.987879  0.979172  0.034399
R2            0.187383  0.200768  0.056215
Spearman_Rho  0.566313  0.560807  0.027411

cluster (internal test) performance: 
                 Median      Mean   Std_Dev
RMSE          0.820197  0.820027  0.005147
R2            0.515131  0.515315  0.006078
Spearman_Rho  0.713832  0.715867  0.003369



In [ ]:
def get_preds(models, base_path, split):
    all_preds = []
    for i, model in enumerate(models):
        model_path = os.path.join(base_path, split, model)
        model_dir = pathlib.Path(model_path).parent
        model_prediction = pd.read_csv(f'{model_dir}/censored_test_preds.csv')
        all_preds.append(model_prediction['unscaled_prediction'].rename(str(i)))
    predictions = pd.concat(all_preds, axis=1)
    model_prediction['mean_predictions'] = predictions.mean(axis=1)
    return predictions, model_prediction

from sklearn.metrics import roc_auc_score

def get_performance(y_true, relations, predictions, tolerance=0.5):
    """
    df_true: DataFrame with 'pchembl_value_Mean' and 'relation'
    predictions: DataFrame containing model continuous predictions
    tolerance: Margin of error for '=' relations (e.g., within 0.5 log units)
    """
    
    metrics_dict = {'Ranking_ROC_AUC': []}
    model_keys = [k for k in predictions.keys() if k.isdigit() or k.startswith('Model')]
    
    for key in model_keys:
        y_pred = predictions[key].values
        
        # 1. Determine correctness (True/False -> 1/0) based on the relation
        is_correct = np.zeros(len(y_true), dtype=int)
        
        # Greater than constraints
        gt_mask = np.isin(relations, ['>', '>=', '>;>=', '>=;>'])
        is_correct[gt_mask & (y_pred > y_true)] = 1
        # Less than constraints
        lt_mask = np.isin(relations, ['<', '<=', '<;<=', '<=;<'])
        is_correct[lt_mask & (y_pred < y_true)] = 1
        
        # Exact match constraints (within an acceptable experimental error tolerance)
        eq_mask = (relations == '=')
        is_correct[eq_mask & (np.abs(y_pred - y_true) <= tolerance)] = 1
        
        # 2. Score = Distance from the cutoff (higher distance = higher model certainty)
        certainty_score = np.abs(y_pred - y_true)
        
        # 3. Compute ROC AUC: Does higher certainty correlate with being correct?
        auc = roc_auc_score(is_correct, certainty_score)
        metrics_dict['Ranking_ROC_AUC'].append(auc)
        
    # Generate summary stats
    df_metrics = pd.DataFrame(metrics_dict, index=[f'Model_{k}' for k in model_keys])
    return pd.DataFrame({
        'Mean': df_metrics.mean(),
        'Median': df_metrics.median(),
        'Std_Dev': df_metrics.std()
    })

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

sns.set_context("talk")

fig, ax = plt.subplots(1, 2, figsize=(15, 6.5), sharey=True)

# First plot
rho1, _ = spearmanr(base_rank_saif_train_mean['mean_predictions'], base_rank_saif_train_mean['percent_inhibition'])
sns.kdeplot(
    data=base_rank_saif_train_mean,
    x='mean_predictions',
    y='percent_inhibition',
    fill=True,
    alpha=0.35,
    linewidth=1,
    ax=ax[0]
)
ax[0].text(
    0.05, 0.95,
    f"Spearman's $\\rho$: {rho1:.2f}",
    transform=ax[0].transAxes,
    va='top', ha='left',
    fontsize=14,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.7, edgecolor='none')
)
ax[0].set_xlabel('predicted pChEMBL')
ax[0].set_ylabel('true % inhibition')
ax[0].set_title('base - ranked')
ax[0].set_xlim(2, 9.0)
ax[0].grid(True, axis='x', linestyle=':', alpha=0.3, color='grey')
ax[0].grid(True, axis='y', linestyle=':', alpha=0.3, color='grey')
# Second plot
rho2, _ = spearmanr(ext_rank_cont_saif_train_mean['mean_predictions'], ext_rank_cont_saif_train_mean['percent_inhibition'])
sns.kdeplot(
    data=ext_rank_cont_saif_train_mean,
    x='mean_predictions',
    y='percent_inhibition',
    fill=True,
    alpha=0.35,
    linewidth=1,
    ax=ax[1]
)
ax[1].text(
    0.05, 0.95,
    f"Spearman's $\\rho$: {rho2:.2f}",
    transform=ax[1].transAxes,
    va='top',
    ha='left',
    fontsize=14,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.7, edgecolor='none')
)
ax[1].set_xlabel('predicted pChEMBL')
ax[1].set_ylabel('')
ax[1].set_title('extended - ranked')
ax[1].set_xlim(2, 9.0)
ax[1].grid(True, axis='x', linestyle=':', alpha=0.3, color='grey')
ax[1].grid(True, axis='y', linestyle=':', alpha=0.3, color='grey')

plt.tight_layout()
# plt.savefig('distribution_perc_inh_pchembl_preds.png', dpi=250)
plt.show()

In [ ]:
ext_rank_cont_saif_train_mean[(ext_rank_cont_saif_train_mean['percent_inhibition'] <20) &  (ext_rank_cont_saif_train_mean['mean_predictions'] > 7)].mut.unique()